# NexaFlow — Análisis con Apache Spark

Procesa los datasets de ventas y reservas de NexaFlow y genera 5 métricas de negocio.

| # | Métrica |
|---|---|
| 1 | Revenue y ventas por tenant |
| 2 | Top productos por cantidad vendida |
| 3 | Tasa de cancelación de reservas por tenant |
| 4 | Reservas por franja horaria |
| 5 | Ventas completadas por mes |

> **Requisito:** los archivos `sales.csv` y `reservations.csv` deben estar en la misma carpeta que este notebook.

## 0 · Instalación de dependencias

In [2]:
import importlib, subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for _pkg in ('pyspark',):
    if importlib.util.find_spec(_pkg) is None:
        print(f'Instalando {_pkg}...')
        _install(_pkg)

print('Dependencias listas.')

Dependencias listas.


## 1 · Inicializar Spark y cargar datos

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("NexaFlow-Analytics")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

sales        = spark.read.csv("sales.csv",        header=True, inferSchema=True)
reservations = spark.read.csv("reservations.csv", header=True, inferSchema=True)

print(f"Dataset: {sales.count()} ventas | {reservations.count()} reservas")
sales.printSchema()

Dataset: 2000 ventas | 1000 reservas
root
 |-- sale_id: string (nullable = true)
 |-- tenant_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total: double (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)



## Métrica 1 · Revenue y ventas por tenant

In [4]:
(sales.filter(F.col("status") == "completed")
      .groupBy("tenant_id")
      .agg(
          F.count("sale_id").alias("total_ventas"),
          F.round(F.sum("total"), 2).alias("revenue_total"),
          F.round(F.avg("total"), 2).alias("ticket_promedio"),
      )
      .orderBy(F.desc("revenue_total"))
      .show(truncate=False))

+------------------------------------+------------+-------------+---------------+
|tenant_id                           |total_ventas|revenue_total|ticket_promedio|
+------------------------------------+------------+-------------+---------------+
|3083fa7f-9968-479f-960f-6cef2a12cd2c|151         |6240.47      |41.33          |
|6be25e64-c34e-4cd9-b91a-42715b89b119|142         |5961.96      |41.99          |
|93327782-e082-4124-826e-9cef7e076d22|139         |5554.75      |39.96          |
|23aa4bfb-faa3-4dec-950e-0dedf62b0788|139         |5400.34      |38.85          |
|3483fce0-915d-4894-a0e2-4ba706fbcf9a|118         |4874.9       |41.31          |
+------------------------------------+------------+-------------+---------------+



## Métrica 2 · Top productos por cantidad vendida

In [5]:
(sales.filter(F.col("status") == "completed")
      .groupBy("product")
      .agg(
          F.sum("quantity").alias("unidades_vendidas"),
          F.round(F.sum("total"), 2).alias("revenue"),
      )
      .orderBy(F.desc("unidades_vendidas"))
      .show(truncate=False))

+--------+-----------------+-------+
|product |unidades_vendidas|revenue|
+--------+-----------------+-------+
|Jugo    |308              |4140.93|
|T�      |279              |3906.21|
|Ensalada|262              |3614.91|
|Pasta   |261              |4017.99|
|Postre  |253              |3491.46|
|Sandwich|248              |3365.53|
|Caf�    |210              |2816.9 |
|Pizza   |203              |2678.49|
+--------+-----------------+-------+



## Métrica 3 · Tasa de cancelación de reservas por tenant

In [6]:
total_res = reservations.groupBy("tenant_id").agg(F.count("*").alias("total"))
cancelled = (reservations.filter(F.col("status") == "cancelled")
             .groupBy("tenant_id")
             .agg(F.count("*").alias("canceladas")))

(total_res.join(cancelled, "tenant_id", "left")
          .fillna(0, subset=["canceladas"])
          .withColumn("tasa_cancelacion_%",
                      F.round(F.col("canceladas") / F.col("total") * 100, 1))
          .orderBy(F.desc("tasa_cancelacion_%"))
          .show(truncate=False))

+------------------------------------+-----+----------+------------------+
|tenant_id                           |total|canceladas|tasa_cancelacion_%|
+------------------------------------+-----+----------+------------------+
|23aa4bfb-faa3-4dec-950e-0dedf62b0788|200  |46        |23.0              |
|6be25e64-c34e-4cd9-b91a-42715b89b119|193  |39        |20.2              |
|3083fa7f-9968-479f-960f-6cef2a12cd2c|224  |45        |20.1              |
|3483fce0-915d-4894-a0e2-4ba706fbcf9a|201  |36        |17.9              |
|93327782-e082-4124-826e-9cef7e076d22|182  |32        |17.6              |
+------------------------------------+-----+----------+------------------+



## Métrica 4 · Reservas por franja horaria

In [7]:
(reservations
      .withColumn("hora", F.split(F.col("time_slot"), ":")[0].cast("int"))
      .withColumn("franja",
          F.when(F.col("hora") < 12, "Mañana (8-12)")
           .when(F.col("hora") < 17, "Tarde (12-17)")
           .otherwise("Noche (17-22)"))
      .groupBy("franja")
      .agg(F.count("*").alias("reservas"))
      .orderBy("franja")
      .show(truncate=False))

+-------------+--------+
|franja       |reservas|
+-------------+--------+
|Noche (17-22)|1000    |
+-------------+--------+



## Métrica 5 · Ventas completadas por mes

In [8]:
(sales.filter(F.col("status") == "completed")
      .withColumn("mes", F.date_format(F.col("created_at"), "yyyy-MM"))
      .groupBy("mes")
      .agg(
          F.count("sale_id").alias("ventas"),
          F.round(F.sum("total"), 2).alias("revenue"),
      )
      .orderBy("mes")
      .show(truncate=False))

spark.stop()
print("Análisis completado.")

+-------+------+-------+
|mes    |ventas|revenue|
+-------+------+-------+
|2024-01|43    |1996.5 |
|2024-02|57    |2058.28|
|2024-03|42    |1613.57|
|2024-04|61    |2707.81|
|2024-05|61    |2684.95|
|2024-06|55    |1981.99|
|2024-07|54    |2011.19|
|2024-08|80    |3456.61|
|2024-09|73    |3242.61|
|2024-10|47    |1365.31|
|2024-11|59    |2481.57|
|2024-12|57    |2432.03|
+-------+------+-------+

Análisis completado.
